# 02 — Bulk RNA-seq Differential Expression

**Run on: ARM64** — Uses `scripts/python/download_data.py` output.

**Target:** Fig 5B — MEOX1 in TGFB+IL1B quadrant

iPSC-derived cardiac fibroblasts: Unstim, IL1B, TGFB, TGFB+IL1B (4 replicates each).

## Data paths

Bulk RNA-seq runs (from `scripts/python/download_data.py`): SRR22882084–SRR22882099

In [3]:
# Check data availability (use R kernel)
bulk_runs <- sprintf("SRR228820%02d", 84:99)
n_available <- sum(dir.exists(file.path("../data", bulk_runs)))
cat("Available:", n_available, "/ 16 bulk RNA-seq runs\n")
if (n_available > 0) {
  first <- bulk_runs[dir.exists(file.path("../data", bulk_runs))][1]
  cat("Sample:", first, "->", paste(head(list.files(file.path("../data", first)), 2), collapse = ", "), "\n")
}

Available: 16 / 16 bulk RNA-seq runs
Sample: SRR22882084 -> SRR22882084_1.fastq.gz, SRR22882084_2.fastq.gz 


## Pipeline

1. **Reference:** hg38 FASTA + GTF (download once if missing)
2. **Align:** Rsubread to hg38
3. **Count:** featureCounts (Ensembl gene models)
4. **DE:** limma-voom
5. **Target:** MEOX1 in TGFB+IL1B quadrant (LogFC > 2 for TGFB; increased in TGFB+IL1B vs TGFB alone)

### Reference genome (hg38)

Download once if missing. Ensembl release 110 used here.

In [4]:
# Paths (relative to notebook dir)
data_dir <- file.path("..", "data")
ref_dir <- file.path("..", "data", "reference")
out_dir <- file.path("..", "output", "bulk_rnaseq")
dir.create(ref_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

# Reference files (Ensembl release 110)
ensembl_release <- "110"
fa_url <- paste0("http://ftp.ensembl.org/pub/release-", ensembl_release, "/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz")
gtf_url <- paste0("http://ftp.ensembl.org/pub/release-", ensembl_release, "/gtf/homo_sapiens/Homo_sapiens.GRCh38.", ensembl_release, ".gtf.gz")
fa_gz <- file.path(ref_dir, basename(fa_url))
gtf_gz <- file.path(ref_dir, basename(gtf_url))

# Download if missing (~880 MB FASTA, ~50 MB GTF). Rsubread accepts .gz directly.
# Increase timeout for large files (default 60s is too short)
options(timeout = 7200)  # 2 hours

# Remove incomplete partial download if present
expected_fa_size <- 881e6  # ~881 MB
if (file.exists(fa_gz) && file.info(fa_gz)$size < expected_fa_size) {
  message("Removing incomplete FASTA (", round(file.info(fa_gz)$size/1e6, 1), " MB)")
  unlink(fa_gz)
}

if (!file.exists(fa_gz)) {
  message("Downloading hg38 FASTA (~880 MB, may take 10-30 min)...")
  download.file(fa_url, fa_gz, mode = "wb")
}
if (!file.exists(gtf_gz)) {
  message("Downloading hg38 GTF...")
  download.file(gtf_url, gtf_gz, mode = "wb")
}

fa_path <- fa_gz
gtf_path <- gtf_gz
cat("FASTA:", fa_path, "exists:", file.exists(fa_path), "\n")
cat("GTF:", gtf_path, "exists:", file.exists(gtf_path), "\n")

Removing incomplete FASTA (273.5 MB)





FASTA: ../data/reference/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz exists: TRUE 
GTF: ../data/reference/Homo_sapiens.GRCh38.110.gtf.gz exists: TRUE 


### Build index and align (Rsubread)

Aligns only runs that have been downloaded. Processes available samples incrementally.

In [5]:
library(Rsubread)

# Index path (build once)
index_prefix <- file.path(ref_dir, "hg38_ens110")
bam_dir <- file.path(out_dir, "bams")
dir.create(bam_dir, recursive = TRUE, showWarnings = FALSE)

# Build index if not present
if (!file.exists(paste0(index_prefix, ".00.b.tab"))) {
  message("Building Rsubread index for hg38...")
  buildindex(basename = index_prefix, reference = fa_path)
}

# Process only available runs
bulk_runs <- sprintf("SRR228820%02d", 84:99)
available <- bulk_runs[dir.exists(file.path(data_dir, bulk_runs))]
cat("Aligning", length(available), "/ 16 samples\n")

for (run in available) {
  run_dir <- file.path(data_dir, run)
  fq1 <- file.path(run_dir, paste0(run, "_1.fastq.gz"))
  fq2 <- file.path(run_dir, paste0(run, "_2.fastq.gz"))
  out_bam <- file.path(bam_dir, paste0(run, ".bam"))
  if (!file.exists(fq1) || !file.exists(fq2)) next
  if (file.exists(out_bam)) { cat("Skip (exists):", run, "\n"); next }
  message("Aligning ", run, " ...")
  align(index = index_prefix, readfile1 = fq1, readfile2 = fq2,
        output_file = out_bam, nthreads = parallel::detectCores(), type = "rna")
}

Building Rsubread index for hg38...




        ==========     _____ _    _ ____  _____  ______          _____  
        =====         / ____| |  | |  _ \|  __ \|  ____|   /\   |  __ \ 
          =====      | (___ | |  | | |_) | |__) | |__     /  \  | |  | |
            ====      \___ \| |  | |  _ <|  _  /|  __|   / /\ \ | |  | |
              ====    ____) | |__| | |_) | | \ \| |____ / ____ \| |__| |
        ==========   |_____/ \____/|____/|_|  \_\______/_/    \_\_____/
       Rsubread 2.24.0

//================================= setting ==================================\\
||                                                                            ||
||                Index name : hg38_ens110                                    ||
||               Index space : base space                                     ||
||               Index split : no-split                                       ||
||          Repeat threshold : 100 repeats                                    ||
||              Gapped index : no                   

Aligning SRR22882084 ...




        ==========     _____ _    _ ____  _____  ______          _____  
        =====         / ____| |  | |  _ \|  __ \|  ____|   /\   |  __ \ 
          =====      | (___ | |  | | |_) | |__) | |__     /  \  | |  | |
            ====      \___ \| |  | |  _ <|  _  /|  __|   / /\ \ | |  | |
              ====    ____) | |__| | |_) | | \ \| |____ / ____ \| |__| |
        ==========   |_____/ \____/|____/|_|  \_\______/_/    \_\_____/
       Rsubread 2.24.0

//================================= setting ==================================\\
||                                                                            ||
|| Function      : Read alignment (RNA-Seq)                                   ||
|| Input file 1  : SRR22882084_1.fastq.gz                                     ||
|| Input file 2  : SRR22882084_2.fastq.gz                                     ||
|| Output file   : SRR22882084.bam (BAM)                                      ||
|| Index name    : hg38_ens110                      

Aligning SRR22882085 ...



...

Aligning SRR22882091 ...




        ==========     _____ _    _ ____  _____  ______          _____  
        =====         / ____| |  | |  _ \|  __ \|  ____|   /\   |  __ \ 
          =====      | (___ | |  | | |_) | |__) | |__     /  \  | |  | |
            ====      \___ \| |  | |  _ <|  _  /|  __|   / /\ \ | |  | |
              ====    ____) | |__| | |_) | | \ \| |____ / ____ \| |__| |
        ==========   |_____/ \____/|____/|_|  \_\______/_/    \_\_____/
       Rsubread 2.24.0

//================================= setting ==================================\\
||                                                                            ||
|| Function      : Read alignment (RNA-Seq)                                   ||
|| Input file 1  : SRR22882091_1.fastq.gz                                     ||
|| Input file 2  : SRR22882091_2.fastq.gz                                     ||
|| Output file   : SRR22882091.bam (BAM)                                      ||
|| Index name    : hg38_ens110                      

Aligning SRR22882092 ...




        ==========     _____ _    _ ____  _____  ______          _____  
        =====         / ____| |  | |  _ \|  __ \|  ____|   /\   |  __ \ 
          =====      | (___ | |  | | |_) | |__) | |__     /  \  | |  | |
            ====      \___ \| |  | |  _ <|  _  /|  __|   / /\ \ | |  | |
              ====    ____) | |__| | |_) | | \ \| |____ / ____ \| |__| |
        ==========   |_____/ \____/|____/|_|  \_\______/_/    \_\_____/
       Rsubread 2.24.0

//================================= setting ==================================\\
||                                                                            ||
|| Function      : Read alignment (RNA-Seq)                                   ||
|| Input file 1  : SRR22882092_1.fastq.gz                                     ||
|| Input file 2  : SRR22882092_2.fastq.gz                                     ||
|| Output file   : SRR22882092.bam (BAM)                                      ||
|| Index name    : hg38_ens110                      

ERROR: Error in .stop_quietly(): 


### featureCounts → count matrix

In [ ]:
# Count matrix output
counts_file <- file.path(out_dir, "bulk_counts.txt")

bams <- list.files(bam_dir, pattern = "\\.bam$", full.names = TRUE)
if (length(bams) == 0) {
  message("No BAMs yet. Run alignment cell above first.")
} else {
  message("Counting ", length(bams), " BAMs...")
  fc <- featureCounts(files = bams, annot.ext = gtf_path, isGTFAnnotationFile = TRUE,
                     GTF.featureType = "exon", GTF.attrType = "gene_id",
                     isPairedEnd = TRUE, nthreads = parallel::detectCores())
  counts <- fc$counts
  colnames(counts) <- sub("\\.bam$", "", basename(colnames(counts)))
  write.table(counts, counts_file, sep = "\t", quote = FALSE)
  cat("Saved:", counts_file, "\n")
}

## R: Design and sample metadata

In [ ]:
# Sample metadata for bulk RNA-seq
samples <- data.frame(
  run = paste0("SRR228820", 84:99),
  condition = rep(c("Unstim", "IL1B", "TGFB", "TGFB_IL1B"), each = 4),
  replicate = rep(1:4, 4)
)
print(samples)

           run condition replicate
1  SRR22882084    Unstim         1
2  SRR22882085    Unstim         2
3  SRR22882086    Unstim         3
4  SRR22882087    Unstim         4
5  SRR22882088      IL1B         1
6  SRR22882089      IL1B         2
7  SRR22882090      IL1B         3
8  SRR22882091      IL1B         4
9  SRR22882092      TGFB         1
10 SRR22882093      TGFB         2
11 SRR22882094      TGFB         3
12 SRR22882095      TGFB         4
13 SRR22882096 TGFB_IL1B         1
14 SRR22882097 TGFB_IL1B         2
15 SRR22882098 TGFB_IL1B         3
16 SRR22882099 TGFB_IL1B         4


## DE analysis (limma-voom)

Requires count matrix from featureCounts above. Subsets to samples present in counts.

In [ ]:
library(edgeR)
library(limma)

counts_file <- file.path("..", "output", "bulk_rnaseq", "bulk_counts.txt")
if (!file.exists(counts_file)) {
  message("Run featureCounts cell first to create ", counts_file)
} else {
  counts <- read.delim(counts_file, row.names = 1, check.names = FALSE)
  # Subset to samples in count matrix
  keep <- samples$run %in% colnames(counts)
  if (sum(keep) < 8) {
    message("Need at least 8 samples (2 per condition) for DE. Have ", sum(keep))
  } else {
    samples_sub <- samples[keep, ]
    counts_sub <- counts[, samples_sub$run, drop = FALSE]
    design <- model.matrix(~ 0 + condition, data = samples_sub)
    colnames(design) <- gsub("condition", "", colnames(design))

    dge <- DGEList(counts = counts_sub, samples = samples_sub)
    dge <- calcNormFactors(dge)
    v <- voom(dge, design, plot = TRUE)
    fit <- lmFit(v, design)

    cont <- makeContrasts(TGFB_vs_Unstim = TGFB - Unstim,
                         TGFB_IL1B_vs_TGFB = TGFB_IL1B - TGFB, levels = design)
    fit2 <- contrasts.fit(fit, cont)
    fit2 <- eBayes(fit2)

    tgfb_res <- topTable(fit2, coef = 1, n = Inf)
    tgfb_il1b_res <- topTable(fit2, coef = 2, n = Inf)

    # MEOX1: expect LogFC > 2 in TGFB, positive in TGFB_IL1B vs TGFB
    meox1_tgfb <- tgfb_res["ENSG00000125510", ]  # MEOX1
    meox1_il1b <- tgfb_il1b_res["ENSG00000125510", ]
    cat("MEOX1 TGFB vs Unstim logFC:", round(meox1_tgfb$logFC, 2), "\n")
    cat("MEOX1 TGFB+IL1B vs TGFB logFC:", round(meox1_il1b$logFC, 2), "\n")
  }
}